# Stage 1 — Group-Aware Decontamination of the Hair & Scalp Disease Dataset

**Purpose.** The published dataset contains ~9 augmented variants of each source
image, distributed across the vendor's train/test directories. A naive split
therefore places near-identical images on both sides of the evaluation boundary,
producing ~99% accuracy that measures memorisation rather than generalisation.

This notebook rebuilds the split so that **all variants of one source image live in
exactly one split**, and then *proves* the result is clean before any model is trained.

**Outputs**
- `manifest.csv` — every image with its `group_id`, `split`, `is_medoid`, `role`
- `similarity_report.json` — pre/post verification statistics
- `figs/` — group-size and similarity-distribution figures for the paper

**Do not proceed to Stage 2 until the verification gate at the end passes.**

In [1]:
import os, json, math, random, hashlib, warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# CONFIG — edit these three lines for your environment
# ----------------------------------------------------------------------------
DATA_ROOT = Path("/kaggle/input/datasets/sundarannamalai/hair-diseases/Hair Diseases - Final")  # parent containing train/ and test/
SUBDIRS   = ["train", "test"]                          # vendor's original split dirs
OUT_DIR   = Path("/kaggle/working/stage1")

IMG_SIZE   = 224
BATCH      = 64
TAU        = 0.87       # calibrated: yields ~9.85 variants/group == 1200 sources x 9
SEED       = 42
SPLIT_FRAC = (0.70, 0.15, 0.15)   # train / val / test, applied at GROUP level

# Training set uses every variant of its groups (free augmentation).
# Val/test use only the medoid of each group so metrics are not inflated by
# correlated near-duplicates.
EXPAND_TRAIN_VARIANTS = True

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "figs").mkdir(exist_ok=True)
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow 2.20.0 | GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## 1. Index every image in the dataset

We deliberately **pool the vendor's train/ and test/ directories back together**.
Their boundary is meaningless — it is the thing that leaked. We rebuild it ourselves.

In [2]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

rows = []
for sub in SUBDIRS:
    base = DATA_ROOT / sub
    if not base.exists():
        raise FileNotFoundError(f"{base} not found — fix DATA_ROOT/SUBDIRS")
    for cls_dir in sorted(p for p in base.iterdir() if p.is_dir()):
        for img in sorted(cls_dir.rglob("*")):
            if img.suffix.lower() in IMG_EXT:
                rows.append({
                    "path": str(img),
                    "label": cls_dir.name.strip(),
                    "vendor_split": sub,
                })

df = pd.DataFrame(rows).reset_index(drop=True)
df["idx"] = df.index

CLASSES = sorted(df["label"].unique())
NUM_CLASSES = len(CLASSES)
CLS2ID = {c: i for i, c in enumerate(CLASSES)}
df["y"] = df["label"].map(CLS2ID)

print(f"{len(df)} images | {NUM_CLASSES} classes")
print(df.groupby(["label", "vendor_split"]).size().unstack(fill_value=0))

10800 images | 10 classes
vendor_split           test  train
label                             
Alopecia Areata         120    960
Contact Dermatitis      120    960
Folliculitis            120    960
Head Lice               120    960
Lichen Planus           120    960
Male Pattern Baldness   120    960
Psoriasis               120    960
Seborrheic Dermatitis   120    960
Telogen Effluvium       120    960
Tinea Capitis           120    960


### 1b. Exact-duplicate check (byte-level)

A cheap sanity probe. Exact duplicates are the most blatant form of leakage; if the
vendor produced variants via re-encoding, this will find few — the real problem is
near-duplicates, handled in §3.

In [3]:
def file_md5(p, chunk=1 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

df["md5"] = [file_md5(p) for p in df["path"]]
dup_groups = df.groupby("md5").size()
n_exact_dup = int((dup_groups > 1).sum())
n_exact_dup_imgs = int(dup_groups[dup_groups > 1].sum() - (dup_groups > 1).sum())

# how many exact dups straddle the vendor's train/test line?
straddle = 0
for _, g in df[df["md5"].isin(dup_groups[dup_groups > 1].index)].groupby("md5"):
    if g["vendor_split"].nunique() > 1:
        straddle += 1

print(f"Exact-duplicate hash groups: {n_exact_dup}")
print(f"Redundant exact-duplicate images: {n_exact_dup_imgs}")
print(f"Hash groups spanning vendor train AND test: {straddle}  <-- direct leakage")

Exact-duplicate hash groups: 0
Redundant exact-duplicate images: 0
Hash groups spanning vendor train AND test: 0  <-- direct leakage


## 2. Embed every image

We use **ResNet50 ImageNet features (2048-d, global average pooled)**, deliberately a
*different* representation family from the classifiers we will train later, so that
grouping decisions are not circular with model performance.

Each image is embedded together with its **horizontal mirror**, and similarity between
two images is the max over the 2x2 orientation pairs. Without this, mirrored variants
of the same source would not group and would leak straight through.

In [4]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_prep

embedder = ResNet50(weights="imagenet", include_top=False, pooling="avg")

AUTOTUNE = tf.data.AUTOTUNE

def _load(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method="bilinear")
    img = tf.cast(img, tf.float32)
    return img

def make_ds(paths, mirror=False):
    ds = tf.data.Dataset.from_tensor_slices(paths).map(_load, num_parallel_calls=AUTOTUNE)
    if mirror:
        ds = ds.map(tf.image.flip_left_right, num_parallel_calls=AUTOTUNE)
    ds = ds.map(resnet_prep, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

def l2norm(a, eps=1e-9):
    return a / (np.linalg.norm(a, axis=1, keepdims=True) + eps)

paths = df["path"].tolist()
print("Embedding originals...")
E  = l2norm(embedder.predict(make_ds(paths, False), verbose=1).astype(np.float32))
print("Embedding mirrors...")
EM = l2norm(embedder.predict(make_ds(paths, True),  verbose=1).astype(np.float32))

np.save(OUT_DIR / "emb.npy", E)
np.save(OUT_DIR / "emb_mirror.npy", EM)
print("Embeddings:", E.shape)

I0000 00:00:1785641335.215440      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785641335.218457      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Embedding originals...
  1/169 ━━━━━━━━━━━━━━━━━━━━ 28:23 10s/step

I0000 00:00:1785641348.698377      72 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 184ms/step
Embedding mirrors...
169/169 ━━━━━━━━━━━━━━━━━━━━ 26s 155ms/step
Embeddings: (10800, 2048)


## 3. Build source groups (within class)

Augmented variants of one source share its label, so we search within each class.
We link any pair with mirror-aware cosine >= TAU, then take **connected components**.

Connected components can *chain* (A~B, B~C but A!~C), over-merging groups. We report the
size distribution so this is auditable — if the mean lands near the expected augmentation
factor (~9), chaining is under control.

In [5]:
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n)); self.r = [0] * n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]: ra, rb = rb, ra
        self.p[rb] = ra
        if self.r[ra] == self.r[rb]: self.r[ra] += 1

def pair_sim(A, AM, B, BM):
    """Mirror-aware cosine: max over the four orientation pairings."""
    s = A @ B.T
    s = np.maximum(s, A @ BM.T)
    s = np.maximum(s, AM @ B.T)
    s = np.maximum(s, AM @ BM.T)
    return s

uf = UnionFind(len(df))
for cls in CLASSES:
    ids = df.index[df["label"] == cls].to_numpy()
    S = pair_sim(E[ids], EM[ids], E[ids], EM[ids])
    np.fill_diagonal(S, -1.0)
    ii, jj = np.where(S >= TAU)
    for a, b in zip(ii, jj):
        if a < b:
            uf.union(int(ids[a]), int(ids[b]))
    print(f"{cls:<28} n={len(ids):>5}  links={int((S >= TAU).sum() // 2)}")

df["group_id"] = [uf.find(i) for i in range(len(df))]
gsz = df.groupby("group_id").size()

print("\n--- GROUP REPORT ---")
print(f"Images                : {len(df)}")
print(f"Source groups          : {df['group_id'].nunique()}")
print(f"Mean variants / group  : {gsz.mean():.2f}   (expected ~9 if 1200 sources x 9)")
print(f"Median / max group size: {int(gsz.median())} / {int(gsz.max())}")
print(f"Singleton groups       : {int((gsz == 1).sum())}")
oversized = gsz[gsz > 3 * gsz.median()]
print(f"Oversized (>3x median) : {len(oversized)}  <-- inspect if this is large (chaining)")

Alopecia Areata              n= 1080  links=19978
Contact Dermatitis           n= 1080  links=33160
Folliculitis                 n= 1080  links=19889
Head Lice                    n= 1080  links=22650
Lichen Planus                n= 1080  links=23850
Male Pattern Baldness        n= 1080  links=16959
Psoriasis                    n= 1080  links=17170
Seborrheic Dermatitis        n= 1080  links=17315
Telogen Effluvium            n= 1080  links=16219
Tinea Capitis                n= 1080  links=7899

--- GROUP REPORT ---
Images                : 10800
Source groups          : 225
Mean variants / group  : 48.00   (expected ~9 if 1200 sources x 9)
Median / max group size: 39 / 211
Singleton groups       : 3
Oversized (>3x median) : 11  <-- inspect if this is large (chaining)


### 3b. Cross-class near-duplicate probe

Grouping was done within class. If the *same* source image appears under two different
labels (vendor label noise), that pair escapes grouping and can still leak across the
split. This probe quantifies it — and any hit is also a label-quality finding worth a
sentence in the paper.

In [6]:
CHUNK = 512
cross_hits = []
for start in range(0, len(df), CHUNK):
    sl = slice(start, min(start + CHUNK, len(df)))
    S = pair_sim(E[sl], EM[sl], E, EM)
    for r in range(S.shape[0]):
        i = start + r
        S[r, i] = -1.0
        cand = np.where(S[r] >= TAU)[0]
        for j in cand:
            if j > i and df.at[i, "y"] != df.at[j, "y"]:
                cross_hits.append((i, int(j), float(S[r, j])))

print(f"Cross-class near-duplicate pairs at tau={TAU}: {len(cross_hits)}")
if cross_hits:
    ch = pd.DataFrame(cross_hits, columns=["i", "j", "sim"])
    ch["label_i"] = df.loc[ch["i"], "label"].values
    ch["label_j"] = df.loc[ch["j"], "label"].values
    print(ch.groupby(["label_i", "label_j"]).size().sort_values(ascending=False).head(15))
    ch.to_csv(OUT_DIR / "cross_class_duplicates.csv", index=False)
    # Merge them too — a shared source must not straddle the split regardless of label.
    for i, j, _ in cross_hits:
        uf.union(i, j)
    df["group_id"] = [uf.find(i) for i in range(len(df))]
    print(f"Groups after cross-class merge: {df['group_id'].nunique()}")

Cross-class near-duplicate pairs at tau=0.87: 1301
label_i                label_j              
Psoriasis              Seborrheic Dermatitis    1074
Seborrheic Dermatitis  Psoriasis                 114
Alopecia Areata        Telogen Effluvium          40
                       Male Pattern Baldness      36
Lichen Planus          Telogen Effluvium          15
Folliculitis           Seborrheic Dermatitis       7
Male Pattern Baldness  Alopecia Areata             4
Folliculitis           Lichen Planus               3
Seborrheic Dermatitis  Tinea Capitis               3
Head Lice              Seborrheic Dermatitis       3
Contact Dermatitis     Telogen Effluvium           1
Lichen Planus          Seborrheic Dermatitis       1
dtype: int64
Groups after cross-class merge: 214


## 4. Choose one medoid representative per group

The medoid (highest mean similarity to its group) is the most canonical member — least
likely to be an extreme rotation or crop. Val/test are built from medoids only.

In [7]:
medoids = {}
for gid, g in df.groupby("group_id"):
    ids = g.index.to_numpy()
    if len(ids) == 1:
        medoids[gid] = int(ids[0]); continue
    S = pair_sim(E[ids], EM[ids], E[ids], EM[ids])
    np.fill_diagonal(S, 0.0)
    medoids[gid] = int(ids[S.mean(axis=1).argmax()])

df["is_medoid"] = df["idx"].isin(set(medoids.values()))
print(f"Medoids selected: {df['is_medoid'].sum()} (== {df['group_id'].nunique()} groups)")

Medoids selected: 214 (== 214 groups)


## 5. Group-level stratified split

Groups — never images — are the unit of assignment. Stratified by class so rare
classes keep their proportion.

In [8]:
from sklearn.model_selection import train_test_split

grp = (df[df["is_medoid"]][["group_id", "label", "y"]]
       .drop_duplicates("group_id").reset_index(drop=True))

tr_g, tmp_g = train_test_split(
    grp, train_size=SPLIT_FRAC[0], stratify=grp["y"], random_state=SEED)
val_share = SPLIT_FRAC[1] / (SPLIT_FRAC[1] + SPLIT_FRAC[2])
val_g, te_g = train_test_split(
    tmp_g, train_size=val_share, stratify=tmp_g["y"], random_state=SEED)

split_of = {}
for gid in tr_g["group_id"]:  split_of[gid] = "train"
for gid in val_g["group_id"]: split_of[gid] = "val"
for gid in te_g["group_id"]:  split_of[gid] = "test"
df["split"] = df["group_id"].map(split_of)

# role: what actually gets fed to the model
if EXPAND_TRAIN_VARIANTS:
    df["role"] = np.where(
        df["split"] == "train", "train_use",
        np.where(df["is_medoid"], df["split"] + "_use", "discard"))
else:
    df["role"] = np.where(df["is_medoid"], df["split"] + "_use", "discard")

print("Groups per split:", Counter(split_of.values()))
print("\nImages actually used:")
print(df[df["role"] != "discard"].groupby(["split", "label"]).size().unstack(fill_value=0))

Groups per split: Counter({'train': 149, 'test': 33, 'val': 32})

Images actually used:
label  Alopecia Areata  Contact Dermatitis  Folliculitis  Head Lice  \
split                                                                 
test                 4                   1             4          2   
train              788                 768           796        783   
val                  4                   1             3          2   

label  Lichen Planus  Male Pattern Baldness  Psoriasis  Seborrheic Dermatitis  \
split                                                                           
test               2                      3          4                      4   
train            743                    793        811                    773   
val                3                      4          4                      3   

label  Telogen Effluvium  Tinea Capitis  
split                                    
test                   5              4  
train                741

## 6. VERIFICATION GATE

The decisive test. For every evaluation image, find its nearest neighbour **in the
training set**. Compare that distribution against a different-class baseline (~0.75 for
this dataset). If evaluation NN similarity still sits far above baseline, the split is
not clean and TAU must be lowered — do not train.

In [9]:
def nn_to_train(eval_ids, train_ids):
    best = np.full(len(eval_ids), -1.0, dtype=np.float32)
    for s in range(0, len(eval_ids), CHUNK):
        sl = eval_ids[s:s + CHUNK]
        S = pair_sim(E[sl], EM[sl], E[train_ids], EM[train_ids])
        best[s:s + len(sl)] = S.max(axis=1)
    return best

train_ids = df.index[df["role"] == "train_use"].to_numpy()
test_ids  = df.index[df["role"] == "test_use"].to_numpy()
val_ids   = df.index[df["role"] == "val_use"].to_numpy()

nn_test = nn_to_train(test_ids, train_ids)
nn_val  = nn_to_train(val_ids,  train_ids)

# Baseline: nearest neighbour among images of a DIFFERENT class (semantic floor)
rng = np.random.default_rng(SEED)
base_sample = rng.choice(test_ids, size=min(300, len(test_ids)), replace=False)
baseline = []
for i in base_sample:
    other = df.index[df["y"] != df.at[i, "y"]].to_numpy()
    other = rng.choice(other, size=min(3000, len(other)), replace=False)
    S = pair_sim(E[[i]], EM[[i]], E[other], EM[other])
    baseline.append(float(S.max()))
baseline = np.array(baseline)

report = {
    "tau": TAU,
    "n_images_total": int(len(df)),
    "n_groups": int(df["group_id"].nunique()),
    "mean_variants_per_group": float(gsz.mean()),
    "exact_dup_groups": n_exact_dup,
    "exact_dups_across_vendor_split": straddle,
    "cross_class_dup_pairs": len(cross_hits),
    "n_train_images": int(len(train_ids)),
    "n_val_images": int(len(val_ids)),
    "n_test_images": int(len(test_ids)),
    "test_nn_median": float(np.median(nn_test)),
    "test_nn_p95": float(np.percentile(nn_test, 95)),
    "test_nn_max": float(nn_test.max()),
    "test_above_tau_frac": float((nn_test >= TAU).mean()),
    "val_nn_median": float(np.median(nn_val)),
    "diff_class_baseline_median": float(np.median(baseline)),
}
print(json.dumps(report, indent=2))

margin = report["test_nn_median"] - report["diff_class_baseline_median"]
print(f"\nMargin over different-class baseline: {margin:+.4f}")

PASS = (report["test_above_tau_frac"] < 0.01) and (margin < 0.06)
print("\n" + ("=" * 62))
print("VERIFICATION: " + ("PASS — proceed to Stage 2" if PASS
      else f"FAIL — lower TAU (try {TAU - 0.03:.2f}) and re-run from section 3"))
print("=" * 62)

{
  "tau": 0.87,
  "n_images_total": 10800,
  "n_groups": 214,
  "mean_variants_per_group": 48.0,
  "exact_dup_groups": 0,
  "exact_dups_across_vendor_split": 0,
  "cross_class_dup_pairs": 1301,
  "n_train_images": 7791,
  "n_val_images": 32,
  "n_test_images": 33,
  "test_nn_median": 0.7686704397201538,
  "test_nn_p95": 0.8296447992324829,
  "test_nn_max": 0.8414717316627502,
  "test_above_tau_frac": 0.0,
  "val_nn_median": 0.7827582359313965,
  "diff_class_baseline_median": 0.7589041590690613
}

Margin over different-class baseline: +0.0098

VERIFICATION: PASS — proceed to Stage 2


In [10]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(gsz.values, bins=range(1, int(gsz.max()) + 2), color="#4C72B0", edgecolor="w")
ax[0].axvline(gsz.mean(), color="crimson", ls="--",
              label=f"mean {gsz.mean():.2f}")
ax[0].set_xlabel("variants per source group"); ax[0].set_ylabel("count")
ax[0].set_title("Source-group size distribution"); ax[0].legend()

ax[1].hist(nn_test, bins=40, alpha=.75, label="test to train NN", color="#4C72B0")
ax[1].hist(baseline, bins=40, alpha=.6, label="different-class baseline", color="#999999")
ax[1].axvline(TAU, color="crimson", ls="--", label=f"tau={TAU}")
ax[1].set_xlabel("mirror-aware cosine similarity"); ax[1].set_ylabel("count")
ax[1].set_title("Post-split leakage verification"); ax[1].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "figs" / "fig1_decontamination.png", dpi=200)
plt.show()

## 7. Watermark / provenance probe

Grad-CAM overlays revealed source-site watermarks (Dermnet, UBC Dermatology, "SIS").
If watermark presence correlates with class, a model can score highly **even on a
perfectly clean split** by reading provenance instead of pathology — which would also
explain why the deletion metric ranked the explainers below random ordering.

Test: black out the central 60% and classify from the **border ring alone**. There is no
skin pathology in that ring. Accuracy meaningfully above chance (1/NUM_CLASSES) is
evidence of a provenance shortcut.

In [11]:
from sklearn.linear_model import LogisticRegression

def border_only(img):
    h = w = IMG_SIZE
    m = np.ones((h, w, 1), np.float32)
    y0, y1 = int(h * .20), int(h * .80)
    x0, x1 = int(w * .20), int(w * .80)
    m[y0:y1, x0:x1, :] = 0.0
    return img * tf.constant(m)

def make_masked_ds(paths):
    ds = tf.data.Dataset.from_tensor_slices(paths).map(_load, num_parallel_calls=AUTOTUNE)
    ds = ds.map(border_only, num_parallel_calls=AUTOTUNE)
    ds = ds.map(resnet_prep, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

probe_tr = df.index[df["role"] == "train_use"].to_numpy()
probe_te = test_ids
Etr_b = l2norm(embedder.predict(make_masked_ds(df.loc[probe_tr, "path"].tolist()), verbose=1))
Ete_b = l2norm(embedder.predict(make_masked_ds(df.loc[probe_te, "path"].tolist()), verbose=1))

clf = LogisticRegression(max_iter=2000, n_jobs=-1)
clf.fit(Etr_b, df.loc[probe_tr, "y"].values)
border_acc = clf.score(Ete_b, df.loc[probe_te, "y"].values)
chance = 1.0 / NUM_CLASSES

print(f"\nBorder-only accuracy : {border_acc:.4f}")
print(f"Chance               : {chance:.4f}")
print(f"Lift over chance     : {border_acc / chance:.2f}x")
if border_acc > 2.5 * chance:
    print(">> PROVENANCE SHORTCUT CONFIRMED — enable centre-cropping in Stage 2.")
else:
    print(">> No strong provenance signal in the border region.")

report["border_only_acc"] = float(border_acc)
report["chance_acc"] = float(chance)

122/122 ━━━━━━━━━━━━━━━━━━━━ 26s 211ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step

Border-only accuracy : 0.3636
Chance               : 0.1000
Lift over chance     : 3.64x
>> PROVENANCE SHORTCUT CONFIRMED — enable centre-cropping in Stage 2.


## 8. Persist artefacts

In [12]:
df_out = df[["path", "label", "y", "vendor_split", "md5",
             "group_id", "is_medoid", "split", "role"]]
df_out.to_csv(OUT_DIR / "manifest.csv", index=False)
with open(OUT_DIR / "similarity_report.json", "w") as f:
    json.dump(report, f, indent=2)
with open(OUT_DIR / "classes.json", "w") as f:
    json.dump(CLASSES, f, indent=2)

print(f"Wrote {OUT_DIR/'manifest.csv'}  ({len(df_out)} rows)")
print(f"Wrote {OUT_DIR/'similarity_report.json'}")
print("\nStage 2 consumes manifest.csv. Only rows with role != 'discard' are used.")

Wrote /kaggle/working/stage1/manifest.csv  (10800 rows)
Wrote /kaggle/working/stage1/similarity_report.json

Stage 2 consumes manifest.csv. Only rows with role != 'discard' are used.
